run operation on a given shape file, 
operation will be run for each sjape separately 
- problems will be reported, then you can edit problematic shapes in QGIS

In [ ]:
# %pip install geopandas

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 11.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 11.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 5.5 MB/s  0:00:00 eta 0:00:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.8 MB/s  0:00:00
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [geopandas]/7 [geopandas]
Note: you may need to restart the kernel to use updated packages.


In [7]:
import geopandas as gpd
import subprocess
import tempfile
import os
from pathlib import Path

# --- INPUTS: adapt these paths to your system ---
vector_path = Path(
    "/Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/"
    ".shortcut-targets-by-id/1p4PDClIbV5Br-EkGzuvhH5WVU1BudYkH/"
    "SLK CorrelAid 2025/QC effis_HRLT_FT_labels_js/"
    "EFFIS_Kosovo_2022_2025_dissolved_clipped_editManual.gpkg"
)
layer_name = "EFFIS_Kosovo_2022_2025_dissolved_clipped_editManual"

raster_path = Path(
    "/Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/"
    ".shortcut-targets-by-id/1p4PDClIbV5Br-EkGzuvhH5WVU1BudYkH/"
    "SLK CorrelAid 2025/QC corine_forest_surface_gfc_overlay/"
    "forest_mask_corine2018.tif"
)

# Where to store per-shape outputs
output_dir = Path("/tmp/effis_per_shape")  # change if you like
output_dir.mkdir(parents=True, exist_ok=True)

# --- Load the vector data ---
gdf = gpd.read_file(vector_path, layer=layer_name)
print(f"Loaded {len(gdf)} features")

failed_features = []

# Optional: choose an attribute to identify features, fallback to index
id_field = None
for candidate in ["fid"]:
    if candidate in gdf.columns:
        id_field = candidate
        break

print("Using feature identifier:", id_field if id_field else "DataFrame index")

# --- Loop over features ---
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    for idx, row in gdf.iterrows():
        # Human-readable ID for logs
        feature_id = row[id_field] if id_field else idx

        # 1) Save this single feature as its own cutline file
        single_gdf = gdf.iloc[[idx]]
        cutline_path = tmpdir / f"cutline_{feature_id}.gpkg"
        single_gdf.to_file(cutline_path, driver="GPKG")

        # 2) Output file for this feature
        out_tif = output_dir / f"clip_{feature_id}.tif"

        # 3) Build the gdalwarp command (your parameters preserved)
        cmd = [
            "gdalwarp",
            "-overwrite",
            "-of", "GTiff",
            "-tr", "100.0", "-100.0",
            "-tap",
            "-cutline", str(cutline_path),
            # no -cl needed, file has only one layer and one feature
            str(raster_path),
            str(out_tif),
        ]

        print(f"Running gdalwarp for feature {feature_id}...")
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print(f"❌ FAILED for feature {feature_id}")
            print(result.stderr[:500])  # show first part of error
            failed_features.append((feature_id, result.stderr))
        else:
            print(f"✅ OK for feature {feature_id}")

print("\n=== Summary ===")
print(f"Total features: {len(gdf)}")
print(f"Failed: {len(failed_features)}")

if failed_features:
    print("\nProblematic feature IDs:")
    for feature_id, _err in failed_features:
        print(" -", feature_id)


Loaded 597 features
Using feature identifier: DataFrame index
Running gdalwarp for feature 0...
✅ OK for feature 0
Running gdalwarp for feature 1...
✅ OK for feature 1
Running gdalwarp for feature 2...
✅ OK for feature 2
Running gdalwarp for feature 3...
✅ OK for feature 3
Running gdalwarp for feature 4...
✅ OK for feature 4
Running gdalwarp for feature 5...
✅ OK for feature 5
Running gdalwarp for feature 6...
✅ OK for feature 6
Running gdalwarp for feature 7...
✅ OK for feature 7
Running gdalwarp for feature 8...
✅ OK for feature 8
Running gdalwarp for feature 9...
✅ OK for feature 9
Running gdalwarp for feature 10...
✅ OK for feature 10
Running gdalwarp for feature 11...
✅ OK for feature 11
Running gdalwarp for feature 12...
✅ OK for feature 12
Running gdalwarp for feature 13...
✅ OK for feature 13
Running gdalwarp for feature 14...
✅ OK for feature 14
Running gdalwarp for feature 15...
✅ OK for feature 15
Running gdalwarp for feature 16...
✅ OK for feature 16
Running gdalwarp for fe